In [ ]:
from pathlib import Path
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import anndata
from typing import List
import time
import matplotlib.pyplot as plt

In [ ]:
download_base = Path('/Volumes/SP PHD U3/Data- ABC-MF/ABC/ABC_Wang_Merge')
abc_cache = AbcProjectCache.from_cache_dir(download_base)

abc_cache.current_manifest

In [ ]:
abc_cache.list_directories

In [ ]:
abc_cache.list_metadata_files('WHB-10Xv3')
abc_cache.get_directory_metadata_size('WHB-10Xv3')

In [ ]:
allen_ccf_list = abc_cache.get_directory_metadata('WHB-10Xv3')
print("WHB-10Xv3 data files:\n\t", allen_ccf_list)

In [ ]:
cell = abc_cache.get_metadata_dataframe(
    directory='WHB-10Xv3',
    file_name='cell_metadata',
    dtype={'cell_label': str}
)
cell.set_index('cell_label', inplace=True)
print("Number of cells = ", len(cell))
cell.head(5)
cell.columns

In [ ]:
membership = abc_cache.get_metadata_dataframe(
    directory='WHB-taxonomy',
    file_name='cluster_to_cluster_annotation_membership'
)
membership_groupby = membership.groupby(['cluster_alias', 'cluster_annotation_term_set_name'])
membership.head(5)

In [ ]:
term_sets = abc_cache.get_metadata_dataframe(directory='WHB-taxonomy', file_name='cluster_annotation_term_set').set_index('label')
cluster_details = membership_groupby['cluster_annotation_term_name'].first().unstack()
cluster_details = cluster_details[term_sets['name']] # order columns
cluster_details.fillna('Other', inplace=True)
cluster_details.sort_values(['supercluster', 'cluster', 'subcluster'], inplace=True)
cluster_details.head(5)

In [ ]:
cluster_colors = membership_groupby['color_hex_triplet'].first().unstack()
cluster_colors = cluster_colors[term_sets['name']]
cluster_colors.sort_values(['supercluster', 'cluster', 'subcluster'], inplace=True)
cluster_colors.head(5)

In [ ]:
roi = abc_cache.get_metadata_dataframe(directory='WHB-10Xv3', file_name='region_of_interest_structure_map')
roi.set_index('region_of_interest_label', inplace=True)
roi.rename(columns={'color_hex_triplet': 'region_of_interest_color'},
           inplace=True)
roi.head(5)

In [ ]:
cell_extended = cell.join(cluster_details, on='cluster_alias')
cell_extended = cell_extended.join(cluster_colors, on='cluster_alias', rsuffix='_color')
cell_extended = cell_extended.join(roi[['region_of_interest_color']], on='region_of_interest_label')
cell_extended.head(5)


In [ ]:
def print_column_info(df):
    
    for c in df.columns:
        grouped = df[[c]].groupby(c).count()
        members = ''
        if len(grouped) < 30:
            members = str(list(grouped.index))
        print("Number of unique %s = %d %s" % (c, len(grouped), members))

print_column_info(cell_extended)

In [ ]:
cell_extended.to_csv('cell_extended.csv', index=True)

In [ ]:
def plot_umap(xx, yy, cc=None, val=None, fig_width=8, fig_height=8, cmap=None):

    fig, ax = plt.subplots()
    fig.set_size_inches(fig_width, fig_height)

    if cmap is not None:
        plt.scatter(xx, yy, s=0.5, c=val, marker='.', cmap=cmap)
    elif cc is not None:
        plt.scatter(xx, yy, s=0.5, color=cc, marker='.')
        
    ax.axis('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    
    return fig, ax

neurons_subsampled = cell_extended[cell_extended['feature_matrix_label'] == 'WHB-10Xv3-Neurons'][::10]
non_neurons_subsampled = cell_extended[cell_extended['feature_matrix_label'] == 'WHB-10Xv3-Nonneurons'][::10]
print("n neurons to plot:", len(neurons_subsampled))
print("n non-neurons to plot:", len(non_neurons_subsampled))

In [ ]:
fig, ax = plot_umap(neurons_subsampled['x'], neurons_subsampled['y'], cc=neurons_subsampled['supercluster_color'])
res = ax.set_title("Neuron Cell Types: Supercluster")
plt.show()
fig, ax = plot_umap(non_neurons_subsampled['x'], non_neurons_subsampled['y'], cc=non_neurons_subsampled['supercluster_color'])
res = ax.set_title("Non-neuron Cell Types: Supercluster")
plt.show()

In [ ]:
gene = abc_cache.get_metadata_dataframe(directory='WHB-10Xv3', file_name='gene')
gene.set_index('gene_identifier', inplace=True)
print("Number of genes = ", len(gene))
gene.head(5)

In [ ]:
wang_df = pd.read_excel('/Volumes/SP PHD U3/Data- ABC-MF/ABC/ABC_Wang_Merge/Supplementary Table 3.xlsx', header = 1)
print(wang_df.head())

In [ ]:
cluster_dict = wang_df.groupby('Cluster')['Gene'].apply(list).to_dict()
print(cluster_dict)

cluster0_genes = cluster_dict[0]
print("Cluster0 Genes:", cluster0_genes)


In [ ]:
wang_df_annotations = pd.read_excel('/Volumes/SP PHD U3/Data- ABC-MF/ABC/ABC_Wang_Merge/Supplementary Table 2.xlsx', header = 1)

In [ ]:
from tabulate import tabulate
columns_to_extract = ['Cluster', 'Type']  
cluster_annotations = wang_df_annotations[columns_to_extract]
type_cluster_mapping = cluster_annotations.groupby('Type')['Cluster'].apply(list).reset_index()
print(tabulate(cluster_annotations, headers='keys', tablefmt='pretty', showindex=False))

In [ ]:
def get_clusters_for_type(type_name):
    clusters = cluster_annotations.loc[cluster_annotations['Type'] == type_name, 'Cluster'].tolist()
    unique_clusters = sorted(set(clusters))
    return unique_clusters


rg_vrg_clusters = get_clusters_for_type("RG-vRG")
rg_trg_clusters = get_clusters_for_type("RG-tRG")
rg_org_clusters = get_clusters_for_type("RG-oRG")
ipc_en_clusters = get_clusters_for_type("IPC-EN")
en_newborn_clusters = get_clusters_for_type("EN-Newborn")
en_it_immature_clusters = get_clusters_for_type("EN-IT-Immature")
en_l2_3_it_clusters = get_clusters_for_type("EN-L2_3-IT")
en_l4_it_clusters = get_clusters_for_type("EN-L4-IT")
en_l5_it_clusters = get_clusters_for_type("EN-L5-IT")
en_l6_it_clusters = get_clusters_for_type("EN-L6-IT")
en_l5_et_clusters = get_clusters_for_type("EN-L5-ET")
en_non_it_immature_clusters = get_clusters_for_type("EN-Non-IT-Immature")
en_l5_6_np_clusters = get_clusters_for_type("EN-L5_6-NP")
en_l6_ct_clusters = get_clusters_for_type("EN-L6-CT")
en_l6b_clusters = get_clusters_for_type("EN-L6b")

print(f"Clusters for type '{'RG-vRG'}': {rg_vrg_clusters}")
print(f"Clusters for type '{'RG-tRG'}': {rg_trg_clusters}")
print(f"Clusters for type '{'RG-oRG'}': {rg_org_clusters}")
print(f"Clusters for type '{'IPC-EN'}': {ipc_en_clusters}")
print(f"Clusters for type '{'EN-Newborn'}': {en_newborn_clusters}")
print(f"Clusters for type '{'EN-IT-Immature'}': {en_it_immature_clusters}")
print(f"Clusters for type '{'EN-L2_3-IT'}': {en_l2_3_it_clusters}")
print(f"Clusters for type '{'EN-L4-IT'}': {en_l4_it_clusters}")
print(f"Clusters for type '{'EN-L5-IT'}': {en_l5_it_clusters}")
print(f"Clusters for type '{'EN-L6-IT'}': {en_l6_it_clusters}")
print(f"Clusters for type '{'EN-L5-ET'}': {en_l5_et_clusters}")
print(f"Clusters for type '{'EN-Non-IT-Immature'}': {en_non_it_immature_clusters}")
print(f"Clusters for type '{'EN-L5_6-NP'}': {en_l5_6_np_clusters}")
print(f"Clusters for type '{'EN-L6-CT'}': {en_l6_ct_clusters}")
print(f"Clusters for type '{'EN-L6b'}': {en_l6b_clusters}")

In [ ]:
def get_genes_for_clusters(cluster_dict, clusters):
    genes = []
    for cluster in clusters:
        genes.extend(cluster_dict.get(cluster, []))
    return sorted(set(genes))

rg_vrg_genes = get_genes_for_clusters(cluster_dict, rg_vrg_clusters)
rg_trg_genes = get_genes_for_clusters(cluster_dict, rg_trg_clusters)
rg_org_genes = get_genes_for_clusters(cluster_dict, rg_org_clusters)
ipc_en_genes = get_genes_for_clusters(cluster_dict, ipc_en_clusters)
en_newborn_genes = get_genes_for_clusters(cluster_dict, en_newborn_clusters)
en_it_immature_genes = get_genes_for_clusters(cluster_dict, en_it_immature_clusters)
en_l2_3_it_genes = get_genes_for_clusters(cluster_dict, en_l2_3_it_clusters)
en_l4_it_genes = get_genes_for_clusters(cluster_dict, en_l4_it_clusters)
en_l5_it_genes = get_genes_for_clusters(cluster_dict, en_l5_it_clusters)
en_l6_it_genes = get_genes_for_clusters(cluster_dict, en_l6_it_clusters)
en_l5_et_genes = get_genes_for_clusters(cluster_dict, en_l5_et_clusters)
en_non_it_immature_genes = get_genes_for_clusters(cluster_dict, en_non_it_immature_clusters)
en_l5_6_np_genes = get_genes_for_clusters(cluster_dict, en_l5_6_np_clusters)
en_l6_ct_genes = get_genes_for_clusters(cluster_dict, en_l6_ct_clusters)
en_l6b_genes = get_genes_for_clusters(cluster_dict, en_l6b_clusters)


In [ ]:
def get_gene_data(
    abc_atlas_cache: AbcProjectCache,
    all_cells: pd.DataFrame,
    all_genes: pd.DataFrame,
    selected_genes: List[str],
    data_type: str = "log2",
    chunk_size: int = 8192
):
    """Load expression matrix data from the ABC Atlas and extract data for
    specific genes.

    Method will load all expression data required to process across multiple
    files to extract the full set of genes. This may result in downloading
    potentially ~100 GB of data.

    Parameters
    ----------
    abc_atlas_cache: AbcProjectCache
        An AbcProjectCache instance object to handle downloading and serving
        the path to the expression matrix data.
    all_cells: pandas.DataFrame
        cells metadata loaded as a pandas Dataframe from the AbcProjectCache
        indexed on cell_label.
    all_genes: pandas.DataFrame
        genes metadata loaded as a pandas Dataframe from the AbcProjectCache
        indexed on gene_identifier.
    selected_genes: list of strings
        List of gene_symbols that are a subset of those in the full genes
        DataFrame.
    data_type: str (Default: "log2")
        Kind of expression matrix to load either "log2" or "raw". Defaults to
        "log2".
    chunk_size: int (Default: 8192)
        Size of the chunk to load from the anndata files. Adjust this size if
        needed based on memory/file io. Default: 8192.

    Returns
    -------
    output_gene_data: pandas.DataFrame
        Subset of gene data indexed by cell.
    """
    # Create a mask for the requested genes.
    gene_mask = np.isin(all_genes.gene_symbol, selected_genes)
    gene_filtered = all_genes[gene_mask]
    # Initialize our output DataFrame.
    output_gene_data = pd.DataFrame(index=all_cells.index,
                                    columns=gene_filtered.index)

    # Get the names of the data in the ABC atlas that we need
    # to load.
    matrices = all_cells.groupby(
        ['dataset_label', 'feature_matrix_label']
    )[['library_label']].count()
    matrices.columns = ['cell_count']

    total_start = time.process_time()
    # Loop over all data files.
    for matrix_index in matrices.index:
        directory = matrix_index[0]
        matrix_file = matrix_index[1]

        print("loading file:", matrix_file)

        file_path = abc_atlas_cache.get_data_path(
            directory=directory,
            file_name=f"{matrix_file}/{data_type}"
        )

        start = time.process_time()
        expression_data = anndata.read_h5ad(file_path, backed='r')
        obs = expression_data.obs
        # Loop over each chunk of the file, slicing by gene
        # and storing the data in the output DataFrame.
        for chunk, min_idx, max_idx in expression_data.chunked_X(
                chunk_size=chunk_size):
            cell_indexes = obs.index[min_idx:max_idx]
            output_gene_data.loc[cell_indexes, gene_filtered.index] = \
                chunk.toarray()[:, gene_mask]

        expression_data.file.close()
        del expression_data  # Clean up our loaded file.
        print(f" - time taken:  {time.process_time() - start}")

    output_gene_data.columns = gene_filtered.gene_symbol
    print(f"total time taken: {time.process_time() - total_start}")
    return output_gene_data

In [ ]:
import scanpy as sc
def aggregate_by_metadata(df, gnames, value, sort = False):
    grouped = df.groupby(value)[gnames].mean()
    if sort:
        grouped = grouped.sort_values(by=gnames[0], ascending=False)
    return grouped

def plot_heatmap(df, fig_width=12, fig_height=4, cmap=plt.cm.magma_r):
    arr = df.to_numpy()

    fig, ax = plt.subplots()
    fig.set_size_inches(fig_width, fig_height)

    im = ax.imshow(arr, cmap=cmap, aspect='auto', vmin=0, vmax=6)
    xlabs = df.columns.values
    ylabs = df.index.values

    ax.set_xticks(range(len(xlabs)))
    ax.set_xticklabels(xlabs, rotation=45, ha='right')

    ax.set_yticks(range(len(ylabs)))
    res = ax.set_yticklabels(ylabs)
    
    return im

list_of_gene_en = {
    "rg_vrg_genes": rg_vrg_genes,
    "rg_trg_genes": rg_trg_genes,
    "rg_org_genes": rg_org_genes,
    "ipc_en_genes": ipc_en_genes,
    "en_newborn_genes": en_newborn_genes,
    "en_it_immature_genes": en_it_immature_genes,
    "en_l2_3_it_genes": en_l2_3_it_genes,
    "en_l4_it_genes": en_l4_it_genes,
    "en_l5_it_genes": en_l5_it_genes,
    "en_l6_it_genes": en_l6_it_genes,
    "en_l5_et_genes": en_l5_et_genes,
    "en_non_it_immature_genes": en_non_it_immature_genes,
    "en_l5_6_np_genes": en_l5_6_np_genes,
    "en_l6_ct_genes": en_l6_ct_genes,
    "en_l6b_genes": en_l6b_genes
}

abc_gene_symbols = set(gene['gene_symbol'])

filtered_list_of_gene_en = {}
missing_gene_log = {}

for group_name, gene_list in list_of_gene_en.items():
    filtered_genes = [gene for gene in gene_list if gene in abc_gene_symbols]
    missing_genes = [gene for gene in gene_list if gene not in abc_gene_symbols]


    if filtered_genes:
        filtered_list_of_gene_en[group_name] = filtered_genes
    if missing_genes:
        missing_gene_log[group_name] = missing_genes
for group, missing in missing_gene_log.items():
    print(f"{group}: Missing genes not in ABC dataset: {missing}")

super_list = list(set(gene for genes in filtered_list_of_gene_en.values() for gene in genes))

target_superclusters = ['Upper-layer intratelencephalic', 'Deep-layer intratelencephalic']

cell_extended_filtered_by_supercluster = cell_extended[
    cell_extended['supercluster'].isin(target_superclusters)
]
print(f"Total genes in super list: {len(super_list)}")
print(f"Total cells in target superclusters: {len(cell_extended_filtered_by_supercluster)}")
print(f"Clusters in these superclusters: {cell_extended_filtered_by_supercluster['cluster'].nunique()}")

In [ ]:
# Now filter neuron_cells to only include cells that are in our filtered cell_extended
neuron_cells = cell[cell['feature_matrix_label'] == 'WHB-10Xv3-Neurons']
neuron_cells_filtered = neuron_cells[neuron_cells.index.isin(cell_extended_filtered_by_supercluster.index)]
print(f"Neuron cells to load: {len(neuron_cells_filtered)}")

In [ ]:
all_gene_data = get_gene_data(
    abc_atlas_cache=abc_cache,
    all_cells=neuron_cells,
    all_genes=gene,
    selected_genes=super_list,
    data_type="log2",
    chunk_size=8192
)

In [ ]:
all_gene_data_filtered = all_gene_data.loc[neuron_cells_filtered.index]
# Check what genes are in super_list vs what we fetched
print(f"Genes in super_list: {len(super_list)}")
print(f"Genes in all_gene_data_filtered columns: {len(all_gene_data_filtered.columns)}")
print(f"Match: {set(super_list) == set(all_gene_data_filtered.columns)}")

# Check which genes are missing
for group_name, gene_list in filtered_list_of_gene_en.items():
    missing = [g for g in gene_list if g not in all_gene_data_filtered.columns]
    if missing:
        print(f"\n{group_name}: {len(missing)} genes missing from fetched data")
        print(f"  Missing: {missing[:10]}...")  # Show first 10

In [ ]:
all_gene_data_filtered = all_gene_data.loc[neuron_cells_filtered.index]

for group_name, gene_list in filtered_list_of_gene_en.items():
    
    print(f"\n{'='*60}")
    print(f"Processing: {group_name}")
    print(f"{'='*60}")
    
    # Only use genes that are actually in the data
    available_genes = [g for g in gene_list if g in all_gene_data_filtered.columns]
    
    if not available_genes:
        print(f"No genes available for {group_name}, skipping...")
        continue
    
    print(f"Using {len(available_genes)}/{len(gene_list)} genes")
    
    gene_data = all_gene_data_filtered[available_genes]
    gene_data = gene_data[pd.notna(gene_data[gene_data.columns[0]])]

    # Get the cell_extended info for these cells
    cell_extended_for_genes = cell_extended_filtered_by_supercluster.loc[gene_data.index]
    cell_extended_with_genes = cell_extended_for_genes.join(gene_data)

    # Aggregate by 'cluster' instead of 'supercluster'
    agg_gene_data = aggregate_by_metadata(cell_extended_with_genes, available_genes, 'cluster')
    
    # Convert to numeric and handle any non-numeric values
    agg_gene_data = agg_gene_data.apply(pd.to_numeric, errors='coerce')
    
    # Check for issues
    print(f"Number of clusters: {len(agg_gene_data)}")
    print(f"Data shape: {agg_gene_data.shape}")
    
    # Check if there's any actual data
    if agg_gene_data.empty or agg_gene_data.isna().all().all():
        print(f"Warning: No valid numeric data for {group_name}, skipping...")
        continue
    
    # Adjust figure height based on number of clusters
    fig_height = max(8, len(agg_gene_data) * 0.3)
    
    # Create new figure for each heatmap
    plt.figure()
    res = plot_heatmap(agg_gene_data, fig_width=20, fig_height=fig_height)
    plt.title(f"{group_name} - Clusters from IT Superclusters") 
    plt.savefig(f"heatmap_{group_name}.pdf", format='pdf', bbox_inches='tight')
    plt.show()
    plt.close()